# Session 2 — Segmentation Without Deep Learning, and the Trap
### Brain MRI Tumour Segmentation · CS Academy Seminar

---

No neural networks today. Not one.

Two things happen this session:

1. You build a tumour segmenter using nothing but **brightness thresholding**, and get a real score.
2. You discover that the way you were scoring it was **lying to you**.

The second one is the important one. It is the single most common mistake in medical AI, it has made
it into published papers, and by the end of today you will never make it.

⏱ Roughly 2 hours.

In [ ]:
#@title Setup — run this first  { display-mode: "form" }
# Downloads the seminar helper code and the dataset.
REPO_RAW = "https://raw.githubusercontent.com/OTMAN-REPO/brain-mri-seminar/main"  #@param {type:"string"}
DATA_URL = ""  #@param {type:"string"}

import os, urllib.request
if not os.path.exists("seminar.py"):
    try:
        urllib.request.urlretrieve(f"{REPO_RAW}/seminar.py", "seminar.py")
        print("Got seminar.py")
    except Exception as e:
        raise SystemExit(f"Could not fetch seminar.py from {REPO_RAW}\n"
                         f"Upload it manually to this Colab session (folder icon on the left).\n{e}")

from seminar import *
import numpy as np, matplotlib.pyplot as plt
images, masks, patient_ids, slice_index = get_data(url=DATA_URL)
print(f"\n{len(images)} slices | {len(np.unique(patient_ids))} patients | image {images.shape[1:]}")
print("device:", DEVICE)

## Part 1 — Segmentation by brightness

Recall from Session 1: on the **FLAIR** channel, tumour tissue glows bright.

So here is a two-line "AI": *call every pixel brighter than some value `t` a tumour.*

That is a real, legitimate baseline. Computer vision did this for decades before deep learning.

In [ ]:
flair = images[..., 1].astype(np.float32)   # channel 1 = FLAIR

j = int(np.argmax(masks.reshape(len(masks), -1).sum(1)))
fig, ax = plt.subplots(1, 5, figsize=(16, 3.4))
ax[0].imshow(flair[j], cmap="gray"); ax[0].set_title("FLAIR")
for k, t in enumerate([100, 140, 180, 220]):
    ax[k+1].imshow(flair[j] > t, cmap="gray"); ax[k+1].set_title(f"brighter than {t}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

ax2 = plt.figure(figsize=(6,2.6)).gca()
ax2.hist(flair[j].ravel(), bins=80, color="grey")
ax2.set_title("brightness histogram of this slice"); ax2.set_xlabel("pixel value")
plt.tight_layout(); plt.show()

### Q1. Pick your threshold

Try a few values and eyeball the result against the radiologist's outline.

In [ ]:
# TODO: change t until the white region looks as close to the green outline as you can get it.
t = 150

fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
ax[0].imshow(flair[j], cmap="gray"); ax[0].set_title("FLAIR")
ax[1].imshow(masks[j], cmap="gray"); ax[1].set_title("radiologist")
ax[2].imshow(flair[j] > t, cmap="gray"); ax[2].set_title(f"your threshold t={t}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

You have probably noticed the problem: **the skull is bright too.**

Eyeballing one slice is not a method. We need a number that says how good a segmentation is,
so we can compare thresholds without squinting.

---

## Part 2 — Scoring, attempt one

The obvious idea: a segmentation is a decision for every pixel, so let's measure
**what fraction of pixels we got right.** That is *pixel accuracy*, and it is the first thing
almost everyone reaches for.

In [ ]:
def pixel_accuracy(pred, true):
    return (np.asarray(pred).astype(bool) == np.asarray(true).astype(bool)).mean()

for t in [100, 140, 180, 220]:
    print(f"threshold {t}:  pixel accuracy = {pixel_accuracy(flair > t, masks):.4%}")

### Q2. Now beat those scores.

Your turn. Build the best model you can, measured by pixel accuracy.

You have five minutes. Whoever gets the highest number wins.

*(Think before you code. There may be a shortcut.)*

In [ ]:
# TODO: build the highest-pixel-accuracy segmenter you can.
#
#       You are allowed to do ANYTHING. There are no rules about what
#       your prediction has to look like.

my_prediction = ...   # an array the same shape as `masks`

print(f"my pixel accuracy = {pixel_accuracy(my_prediction, masks):.4%}")

## 🛑 STOP. Discuss before running the next cell.

Did anyone try predicting **zero everywhere**? No tumour, ever, in any patient?

Run it.

In [ ]:
lazy = np.zeros_like(masks)        # "there is never a tumour, anywhere, in anyone"

print(f"pixel accuracy of the lazy model : {pixel_accuracy(lazy, masks):.4%}")
print()
for t in [100, 140, 180, 220]:
    print(f"  ...vs threshold {t}: {pixel_accuracy(flair > t, masks):.4%}")

### The lazy model wins.

A model that has never heard of a brain, does no computation, and would miss **every tumour in
every patient** scores higher than any honest attempt.

It is not a bug in the code. It is exactly right: from Session 1, roughly **99% of pixels are
background**. Anything that says "background" everywhere is 99% correct.

> Imagine deploying that in a hospital and telling the radiologists it is 99% accurate.

The metric was never measuring what we cared about. We care about **the tumour**, and the metric
was dominated by everything that isn't one.

---

## Part 3 — Scoring, attempt two: Dice

The **Dice similarity coefficient** ignores the background entirely. It asks: *of the tumour pixels
that either of us marked, how much did we agree on?*

$$\text{Dice} = \frac{2 \times |\text{overlap}|}{|\text{prediction}| + |\text{truth}|}$$

- perfect overlap → **1.0**
- no overlap at all → **0.0**
- predicting nothing when there *is* a tumour → **0.0**, because the overlap is zero

Notice what is missing from that formula: any term for pixels you both correctly called background.

In [ ]:
# TODO: implement Dice yourself. Do not use the library version yet.
#
#   overlap = number of pixels where BOTH pred and true are 1
#   total   = (number of 1s in pred) + (number of 1s in true)
#   dice    = 2 * overlap / total

def my_dice(pred, true):
    pred = np.asarray(pred).astype(bool)
    true = np.asarray(true).astype(bool)
    overlap = ...
    total   = ...
    if total == 0:
        return 1.0          # both empty: nothing was there, and we agreed
    return 2 * overlap / total

# check yourself against these
a = np.array([[1,1],[0,0]]); b = np.array([[1,0],[0,0]])
print("expect 1.000 ->", f"{my_dice(a,a):.3f}")
print("expect 0.667 ->", f"{my_dice(a,b):.3f}")
print("expect 0.000 ->", f"{my_dice(a, np.zeros_like(a)):.3f}")
print("expect 1.000 ->", f"{my_dice(np.zeros_like(a), np.zeros_like(a)):.3f}")

### Q3. Now score the lazy model again.

In [ ]:
print(f"lazy model  -> pixel accuracy {pixel_accuracy(lazy, masks):.4%}   Dice {dice_score(lazy, masks):.4f}")
print()
print("Two metrics. Same model. One says 99%, the other says 0.")
print("Only one of them was ever telling you the truth.")

## Part 4 — Now find your real baseline

Sweep the threshold and score it with Dice this time. This number goes on the board, and in
Session 3 your neural network has to beat it.

In [ ]:
ts = np.arange(80, 250, 5)
scores = [dice_score(flair > t, masks) for t in ts]

plt.figure(figsize=(7,3.2))
plt.plot(ts, scores, "o-", ms=3)
plt.xlabel("threshold"); plt.ylabel("Dice"); plt.grid(alpha=.3)
plt.title("threshold sweep, scored properly"); plt.tight_layout(); plt.show()

best_t = ts[int(np.argmax(scores))]
BASELINE_DICE = max(scores)
print(f"best threshold = {best_t}   Dice = {BASELINE_DICE:.4f}")

### Q4. That is probably a low number. Make it better.

You are still not allowed a neural network. But you *are* allowed to be clever.

Ideas worth trying:
- **The skull is the problem.** It is bright, and it is always at the edge. Can you ignore it?
  (Look up `scipy.ndimage.binary_erosion`, or just mask out everything outside a central ellipse.)
- **Tumours are blobs, not speckle.** Remove tiny connected components with `skimage.measure.label`.
- **Brightness is relative.** One scanner's 150 is another's 200. Try normalising each slice by its
  own mean and standard deviation before thresholding.
- `skimage.filters.threshold_otsu` picks a threshold automatically.

In [ ]:
from scipy import ndimage
from skimage import measure

def my_segmenter(flair_stack):
    """TODO: return a binary array the same shape as flair_stack."""
    pred = flair_stack > best_t
    # TODO: remove the skull
    # TODO: remove small blobs
    # TODO: try per-slice normalisation
    return pred

improved = my_segmenter(flair)
print(f"plain threshold : {BASELINE_DICE:.4f}")
print(f"improved        : {dice_score(improved, masks):.4f}")

## Part 5 — One more wrinkle: how do you average?

You have been scoring the whole dataset as one giant pile of pixels. But a hospital does not care
about pixels — it cares about **patients**.

Those give different answers, and the difference is not small.

In [ ]:
global_dice = dice_score(improved, masks)
per_slice   = np.mean([dice_score(improved[k], masks[k]) for k in range(len(masks))])
per_patient = np.mean(list(per_patient_dice(improved, masks, patient_ids).values()))

print(f"pooled over all pixels : {global_dice:.4f}")
print(f"averaged per slice     : {per_slice:.4f}   <- inflated by the empty slices scoring 1.0")
print(f"averaged per patient   : {per_patient:.4f}   <- what a clinician would ask for")
print("\nSame predictions. Three numbers. When you read a paper, check which one they reported.")

---

## Exit ticket

1. Write down your best **Dice**. That is the number to beat in Session 3.
2. In one sentence: why did pixel accuracy fail here?
3. Name one *other* situation — anywhere, not just medicine — where a rare thing would break
   accuracy the same way.
4. The lazy model scored 0 on Dice. Can you invent a *different* cheating model that scores well on
   Dice but is still useless? (Harder than it sounds. Try.)

In [ ]:
#@markdown ### Session 2 exit ticket
my_best_dice = ""  #@param {type:"string"}
why_accuracy_failed = ""  #@param {type:"string"}
another_example_of_this_trap = ""  #@param {type:"string"}
can_dice_be_cheated = ""  #@param {type:"string"}
print("Saved. Session 3: you build the network.")